# 3억 투자자를 위한 3년 후 아파트 가격 예측 실험

## 페르소나

초보 부동산 투자자.

- 투자금: 약 3억 원
- 관심 지역: 서울
- 투자 기간: 3년
- 질문: 3억 원 정도로 접근 가능한 아파트가 3년 후 얼마나 오를까?

주의: 이 실습은 실제 투자 추천이 아니라 머신러닝/서비스 기획 학습용 분석이다.

## 1. 실험 설계

데이터가 2008년 1월부터 2017년 11월까지 있으므로, 과거 시점에서 3년 후를 예측하는 방식으로 검증한다.

첫 실험:

- 기준 시점: 2014년 거래
- 미래 시점: 2017년 거래
- 투자금 기준: 기준 거래가 3억 원 이하 또는 3억 원 근처
- 예측 목표: 3년 후 거래가, 예상 상승액, 예상 상승률

In [ ]:
from pathlib import Path
import warnings

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

warnings.filterwarnings("ignore")

FONT_PATH = "/System/Library/Fonts/AppleSDGothicNeo.ttc"
font_manager.fontManager.addfont(FONT_PATH)
FONT_NAME = font_manager.FontProperties(fname=FONT_PATH).get_name()
rcParams["font.family"] = FONT_NAME
rcParams["axes.unicode_minus"] = False
rcParams["figure.figsize"] = (10, 5)
rcParams["axes.grid"] = True

BASE_DIR = Path("..").resolve()
DATA_PATH = BASE_DIR / "data" / "processed" / "seoul_train_cleaned_baseline.csv"

df = pd.read_csv(DATA_PATH)
df["transaction_price_eok"] = df["transaction_real_price"] / 10000
df["transaction_year"] = df["transaction_year_month"] // 100
df["area_bucket"] = (df["exclusive_use_area"] / 10).round() * 10

df[["transaction_year_month", "transaction_year", "dong", "apt", "exclusive_use_area", "transaction_price_eok"]].head()

## 2. 3억 기준 후보 확인

In [ ]:
investment_limit_manwon = 30000

under_3eok = df[df["transaction_real_price"] <= investment_limit_manwon]

summary = pd.Series({
    "전체 서울 거래 건수": len(df),
    "3억 이하 거래 건수": len(under_3eok),
    "3억 이하 비율(%)": round(len(under_3eok) / len(df) * 100, 2),
    "전체 거래가 중앙값(억 원)": round(df["transaction_price_eok"].median(), 2),
    "3억 이하 거래가 중앙값(억 원)": round(under_3eok["transaction_price_eok"].median(), 2),
})

summary

## 3. 기준 시점과 3년 후 시점 만들기

아파트 개별 거래는 같은 집이 정확히 3년 뒤 다시 거래된다는 보장이 없다.

그래서 첫 실험에서는 다음 단위로 묶어서 비교한다.

- `dong`
- `apt`
- `area_bucket`

즉, 같은 동/단지/면적대의 2014년 중앙 거래가와 2017년 중앙 거래가를 비교한다.

In [ ]:
base_year = 2014
future_year = base_year + 3
group_cols = ["dong", "apt", "area_bucket"]

base_price = (
    df[df["transaction_year"] == base_year]
    .groupby(group_cols)
    .agg(
        base_price_manwon=("transaction_real_price", "median"),
        base_count=("transaction_id", "count"),
        base_area=("exclusive_use_area", "median"),
        base_floor=("floor", "median"),
        base_building_age=("building_age", "median"),
    )
    .reset_index()
)

future_price = (
    df[df["transaction_year"] == future_year]
    .groupby(group_cols)
    .agg(
        future_price_manwon=("transaction_real_price", "median"),
        future_count=("transaction_id", "count"),
    )
    .reset_index()
)

investment_df = base_price.merge(future_price, on=group_cols, how="inner")
investment_df["base_price_eok"] = investment_df["base_price_manwon"] / 10000
investment_df["future_price_eok"] = investment_df["future_price_manwon"] / 10000
investment_df["growth_amount_eok"] = investment_df["future_price_eok"] - investment_df["base_price_eok"]
investment_df["growth_rate_percent"] = (investment_df["future_price_manwon"] / investment_df["base_price_manwon"] - 1) * 100
investment_df["under_3eok"] = investment_df["base_price_manwon"] <= 30000

investment_df.shape

## 4. 3억 이하 후보의 3년 후 상승률 보기

In [ ]:
under_3eok_candidates = investment_df[investment_df["under_3eok"]].copy()

under_3eok_candidates[[
    "base_price_eok",
    "future_price_eok",
    "growth_amount_eok",
    "growth_rate_percent",
    "base_count",
    "future_count",
]].describe().round(2)

In [ ]:
under_3eok_candidates["growth_rate_percent"].clip(-50, 100).hist(bins=60, color="#2563eb")
plt.title("3억 이하 후보의 3년 후 상승률 분포 (2014 -> 2017)")
plt.xlabel("3년 상승률 (%)")
plt.ylabel("후보 수")
plt.show()

## 5. 상승률 상위 후보 보기

거래 수가 너무 적으면 신뢰도가 낮으므로 기준 시점과 미래 시점 모두 3건 이상 거래된 후보만 본다.

In [ ]:
reliable_candidates = under_3eok_candidates[
    (under_3eok_candidates["base_count"] >= 3)
    & (under_3eok_candidates["future_count"] >= 3)
].copy()

top_growth = reliable_candidates.sort_values("growth_rate_percent", ascending=False).head(30)
top_growth[[
    "dong",
    "apt",
    "area_bucket",
    "base_price_eok",
    "future_price_eok",
    "growth_amount_eok",
    "growth_rate_percent",
    "base_count",
    "future_count",
]].round(2)

## 6. Next.js 지도 서비스로 연결할 데이터 형태

지도 서비스에서는 아래와 같은 JSON이 필요하다.

- 아파트명
- 동
- 현재 기준 가격
- 3년 후 예상 가격
- 예상 상승액
- 예상 상승률
- 지도 좌표

현재 데이터에는 좌표가 없으므로 다음 단계에서 주소 기반 좌표 변환이 필요하다.

In [ ]:
map_export_sample = reliable_candidates.sort_values("growth_rate_percent", ascending=False).head(100).copy()
map_export_sample = map_export_sample[[
    "dong",
    "apt",
    "area_bucket",
    "base_price_eok",
    "future_price_eok",
    "growth_amount_eok",
    "growth_rate_percent",
    "base_count",
    "future_count",
]]

map_export_sample.head()